In [51]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go

from IPython.display import IFrame

from utils.benchmark import * 
import tikzplotly


In [52]:
notebook_name="03-geobft-alternating"
os.makedirs(f"outputs/{notebook_name}", exist_ok=True)



In [53]:
folder = "../../aws/benchmark/out/geobft/final-geobft-alternating-opt/"
benchmarks = glob(f"{folder}*/")
folder = "../../aws/benchmark/out/geobft/final-geobft-alternating-non-optimistic/"
benchmarks += glob(f"{folder}*/")
folder="../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/"
benchmarks += glob(f"{folder}*/")

In [54]:


throughputs = process_throughput_benchmarks(benchmarks)
throughputs["files"] = throughputs["benchmark"]
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "not-alternating-non-optimistic" if "not-alternating-non-optimistic" in s else (
    "alternating-non-optimistic" if "alternating-non-optimistic" in s else "alternating-optimistic"
))
dropped =  []

dropped.append(throughputs[((throughputs["benchmark"]=="not-alternating-non-optimistic") & (throughputs["alternating"]!=False))])
dropped.append(throughputs[((throughputs["benchmark"]=="alternating-non-optimistic") & (throughputs["alternating"]!=True))])
dropped.append(throughputs[((throughputs["benchmark"]=="alternating-optimistic") & (throughputs["alternating"]!=True))])
dropped.append(throughputs[((throughputs["benchmark"]=="alternating-optimistic") & (throughputs["nonoptimistic"]!=False))])
dropped.append(throughputs[((throughputs["benchmark"]=="alternating-non-optimistic") & (throughputs["nonoptimistic"]!=True))])
dropped.append(throughputs[((throughputs["benchmark"]=="not-alternating-non-optimistic") & (throughputs["nonoptimistic"]!=True))])

dropped = pd.concat(dropped, ignore_index=True)

# drop where throughputs.query("benchmark == 'not-alternating-non-optimistic'")["alternating"]!=False
throughputs = throughputs[~((throughputs["benchmark"]=="not-alternating-non-optimistic") & (throughputs["alternating"]!=False))]
throughputs = throughputs[~((throughputs["benchmark"]=="alternating-non-optimistic") & (throughputs["alternating"]!=True))]
throughputs = throughputs[~((throughputs["benchmark"]=="alternating-optimistic") & (throughputs["alternating"]!=True))]
throughputs = throughputs[~((throughputs["benchmark"]=="alternating-optimistic") & (throughputs["nonoptimistic"]!=False))]
throughputs = throughputs[~((throughputs["benchmark"]=="alternating-non-optimistic") & (throughputs["nonoptimistic"]!=True))]
throughputs = throughputs[~((throughputs["benchmark"]=="not-alternating-non-optimistic") & (throughputs["nonoptimistic"]!=True))]



assert all(throughputs.query("benchmark == 'not-alternating-non-optimistic'")["alternating"]==False), f"{throughputs.query('benchmark == \"not-alternating-non-optimistic\"')}"
assert all(throughputs.query("benchmark == 'alternating-non-optimistic'")["alternating"]==True), f"{throughputs.query('benchmark == \"alternating-non-optimistic\"')}"
assert all(throughputs.query("benchmark == 'alternating-optimistic'")["alternating"]==True), f"{throughputs.query('benchmark == \"alternating-optimistic\"')}"
assert all(throughputs.query("benchmark == 'alternating-optimistic'")["nonoptimistic"]==False), f"{throughputs.query('benchmark == \"alternating-optimistic\"')}"
assert all(throughputs.query("benchmark == 'alternating-non-optimistic'")["nonoptimistic"]==True), f"{throughputs.query('benchmark == \"alternating-non-optimistic\"')}"
assert all(throughputs.query("benchmark == 'not-alternating-non-optimistic'")["nonoptimistic"]==True), f"{throughputs.query('benchmark == \"not-alternating-non-optimistic\"')}"


throughputs["num_peers"] = throughputs["num_peers"].astype(int)/throughputs["number_of_clusters"].astype(int)

benchmark_names = throughputs["benchmark"].unique().tolist()


batchsizes = [1,16,64,128]
cluster = [2,4,6]
peers  = [3,6]
vallen = [512,4096]

missing = pd.DataFrame(columns=["benchmark","vallen","num_peers","number_of_clusters","cluster_batch_size"])
existing= pd.DataFrame(columns=["benchmark","vallen","num_peers","number_of_clusters","cluster_batch_size","file"])
for v in vallen:
    for p in peers:
        for c in cluster:
            for b in batchsizes:
                
                subset = throughputs.query(f"vallen == {v} and num_peers == {p} and number_of_clusters == {c} and cluster_batch_size == {b}")
                existing_benchmarks = subset["benchmark"].unique().tolist()
                missing_benchmarks = set(benchmark_names) - set(existing_benchmarks)
                for mb in missing_benchmarks:
                    missing = pd.concat([missing, pd.DataFrame([{
                        "benchmark": mb,
                        "vallen": v,
                        "num_peers": p,
                        "number_of_clusters": c,
                        "cluster_batch_size": b,
                    }])], ignore_index=True)
                for eb in existing_benchmarks:
                    existing = pd.concat([existing, pd.DataFrame([{
                        "benchmark": eb,
                        "vallen": v,
                        "num_peers": p,
                        "number_of_clusters": c,
                        "cluster_batch_size": b,
                        "file": subset.query(f"benchmark == '{eb}'")["files"].values[0]
                    }])], ignore_index=True)


In [55]:
print(existing.query("vallen == 512 and number_of_clusters == 6 and cluster_batch_size==128").to_string())

                         benchmark vallen num_peers number_of_clusters cluster_batch_size                                                                                                   file
33          alternating-optimistic    512         3                  6                128                 ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251031132059-cb128-v512/
34      alternating-non-optimistic    512         3                  6                128      ../../aws/benchmark/out/geobft/final-geobft-alternating-non-optimistic/20251028160252-cb128-v512/
35  not-alternating-non-optimistic    512         3                  6                128  ../../aws/benchmark/out/geobft/final-geobft-not-alternating-non-optimistic/20251030140148-cb128-v512/
69          alternating-optimistic    512         6                  6                128                 ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251031163744-cb128-v512/
70      alternating-non-optimistic 

In [56]:
throughputs= throughputs.sort_values("files")
throughputs.drop_duplicates(["benchmark","vallen","num_peers","number_of_clusters","cluster_batch_size"],inplace=True,keep="last")

In [57]:
throughputs.to_json(f"outputs/{notebook_name}/throughputs.json",orient="records")


In [58]:
missing.sort_values(["benchmark","num_peers","number_of_clusters","cluster_batch_size"]).query("cluster_batch_size < 256")

,benchmark,vallen,num_peers,number_of_clusters,cluster_batch_size


In [59]:
throughputs.query("number_of_clusters == 2 and benchmark == 'alternating-non-optimistic'").groupby(["vallen","num_peers","cluster_batch_size"])

In [60]:
grouped = throughputs.query("cluster_batch_size < 256").groupby(["num_peers","number_of_clusters"])
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["cluster_batch_size","vallen","benchmark"]).groupby(["vallen","benchmark"]):
        # different dashes for benchmarks 
        # display(data)
        
        fig.add_trace(go.Scatter(
            name=f"vallen={v[0]} - {v[1]}",
            x=data["cluster_batch_size"],
            y=data["throughput"],
            mode='lines+markers',
            line=dict(dash='solid' if v[1]=="alternating-optimistic" else ('dash' if v[1]=="alternating-non-optimistic" else 'dot')),
            
        ))
    fig.update_layout(
        title=f"GeoBFT - Throughput - numpeers={gr[0]} - numberofclusters={gr[1]}",
        xaxis_title="Cluster Batch Size",
        yaxis_title="Throughput (tx/s)",
        barmode='group'
    )
    fig.show()
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-throughput-numpeers-{gr[0]}-clusterbatchsize-{gr[1]}.tex",fig=fig)    


/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [61]:
grouped = throughputs.query("cluster_batch_size < 256").groupby(["num_peers","cluster_batch_size"])
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["number_of_clusters","vallen","benchmark"]).groupby(["vallen","number_of_clusters"]):
        
        fig.add_trace(go.Scatter(
            name=f"vallen={v[0]} - numclusters={v[1]}",
            x=data["benchmark"],
            y=data["throughput"]
        ))
    fig.update_layout(
        title=f"GeoBFT - Throughput - numpeers={gr[0]} - clusterbatchsize={gr[1]}",
        xaxis_title="Benchmark",
        yaxis_title="Throughput (tx/s)",
        barmode='group'
    )
    fig.show()
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-throughput-numpeers-{gr[0]}-clusterbatchsize-{gr[1]}.tex",fig=fig)

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



In [62]:
processed_cpu_usage = process_cpu_usage(benchmarks)
processed_cpu_usage["num_peers"] = processed_cpu_usage["num_peers"]/processed_cpu_usage["number_of_clusters"]
# processed_cpu_usage.drop_duplicates(["num_peers","vallen","cluster_batch_size","nonoptimistic","alternating","number_of_clusters"], inplace=True,keep="last")


In [63]:
processed_cpu_usage = processed_cpu_usage.set_index("benchmark").loc[throughputs["files"]].reset_index()


In [64]:
processed_cpu_usage["files"] = processed_cpu_usage["benchmark"]
processed_cpu_usage["benchmark"] = processed_cpu_usage["benchmark"].apply(lambda s: "not-alternating-non-optimistic" if "not-alternating-non-optimistic" in s else (
    "alternating-non-optimistic" if "alternating-non-optimistic" in s else "alternating-optimistic"
))
processed_cpu_usage.query("benchmark == 'not-alternating-non-optimistic' and alternating == False").to_json(f"outputs/{notebook_name}/cpu_usage.json",orient="records")

In [65]:
grouped = processed_cpu_usage.query("cluster_batch_size < 256 and role > 0 ").groupby(["num_peers","number_of_clusters","role"])
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["cluster_batch_size","vallen","benchmark"]).groupby(["vallen","benchmark"]):
        # different dashes for benchmarks 
        # display(data)
        
        fig.add_trace(go.Scatter(
            name=f"vallen={v[0]} - {v[1]}",
            x=data.groupby(["benchmark","cluster_batch_size"])["cluster_batch_size"].mean(),
            y=data.groupby(["benchmark","cluster_batch_size"])["cpu_usage"].mean(),
            mode='lines+markers',
            line=dict(dash='solid' if v[1]=="alternating-optimistic" else ('dash' if v[1]=="alternating-non-optimistic" else 'dot')),
            
        ))
    fig.update_layout(
        title=f"GeoBFT - cpu_usage - numpeers={gr[0]} - numberofclusters={gr[1]} - role={gr[2]}",
        xaxis_title="Cluster Batch Size",
        yaxis_title="cpu_usage in percent",
        barmode='group'
    )
    fig.show()
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-cpu-usage-numpeers-{gr[0]}-numclusters-{gr[1]}-role-{gr[2]}.tex",fig=fig)    


/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



...

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [66]:
grouped = processed_cpu_usage.query("cluster_batch_size in [64,128] and role > 0 ").groupby(["num_peers","cluster_batch_size","role"])
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["number_of_clusters","cluster_batch_size","vallen","benchmark"]).groupby(["vallen","number_of_clusters"]):
        # different dashes for benchmarks 
        # display(data)
        
        fig.add_trace(go.Scatter(
            name=f"vallen={v[0]} - {v[1]}",
            x=data.groupby(["benchmark","number_of_clusters"])["benchmark"].apply(lambda x : x.iloc[0]),
            y=data.groupby(["benchmark","number_of_clusters"])["cpu_usage"].mean(),
            mode='lines+markers',
            line=dict(dash='solid' if v[1]=="alternating-optimistic" else ('dash' if v[1]=="alternating-non-optimistic" else 'dot')),
            
        ))
    fig.update_layout(
        title=f"GeoBFT - cpu_usage - numpeers={gr[0]} - numberofclusters={gr[1]} - role={gr[2]}",
        xaxis_title="Cluster Batch Size",
        yaxis_title="cpu_usage in percent",
        barmode='group'
    )
    fig.show()
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-cpuusage-numpeers-{gr[0]}-clusterbatchsize-{gr[1]}-role-{gr[2]}.tex",fig=fig)

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_data.py:21: UserWarning:

Assuming this is a date, add "\usetikzlibrary{pgfplots.dateplot}" to your tex preamble.



In [67]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent = bytes_sent.set_index("benchmark").loc[throughputs["files"]].reset_index()
bytes_sent["files"] = bytes_sent["benchmark"]
bytes_sent["benchmark"] = bytes_sent["benchmark"].apply(lambda s: "not-alternating-non-optimistic" if "not-alternating-non-optimistic" in s else (
    "alternating-non-optimistic" if "alternating-non-optimistic" in s else "alternating-optimistic"
))
bytes_sent = bytes_sent.query("cluster_batch_size < 256 and role > 0 ")
bytes_sent["num_peers"] = bytes_sent["num_peers"]/bytes_sent["number_of_clusters"]
bytes_sent_in_gbps = (bytes_sent.groupby(["num_peers","vallen","role","number_of_clusters","cluster_batch_size","benchmark","files","alternating"])[["bytes_sent"]].sum() * 8 * 10**(-9) * 15**(-1)).reset_index()
bytes_sent_in_gbps["bytes_sent"] = bytes_sent_in_gbps["bytes_sent"] / bytes_sent_in_gbps["number_of_clusters"]
bytes_sent_in_gbps.query("benchmark == 'not-alternating-non-optimistic' and alternating == False").to_json(f"outputs/{notebook_name}/bytes_sent_gbps.json",orient="records")



In [68]:
grouped = bytes_sent_in_gbps.groupby(["num_peers","role"])

In [69]:
for gr, g in grouped:
    fig  = go.Figure()
    for v,data in g.sort_values(["benchmark","cluster_batch_size","vallen","number_of_clusters"]).groupby(["vallen","benchmark","number_of_clusters"]):
        # dashed if payload size is 4096 
        # display(data)
     
        fig.add_trace(go.Scatter(
            name=f"vallen={v[0]} - {v[1]} - numclusters={v[2]}",
            y=data.groupby(["role","cluster_batch_size","number_of_clusters"])['bytes_sent'].mean(),
            x=data.groupby(["role","cluster_batch_size","number_of_clusters"])['cluster_batch_size'].mean(),
            mode='lines+markers',
            line=dict(dash='dash' if v[1]=="alternating-non-optimistic" else ('solid' if v[1]=="alternating-optimistic" else 'dot'))
        ))
    fig.update_layout(
        title=f"GeoBFT Bytes Sent Role: {gr[1]} - numpeers={gr[0]}",
        xaxis_title="Cluster Batch Size",
        yaxis_title="Bytes Sent in Gbps",
        barmode='group'
    )
    tikzplotly.save(filepath=f"outputs/{notebook_name}/geobft-bytes-sent-numpeers-{gr[0]}-role-{gr[1]}-gbps.tex",fig=fig)
    fig.show()

/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



/home/andre/miniconda3/lib/python3.13/site-packages/tikzplotly/_axis.py:149: UserWarning:

The categoryorder option is not supported (yet 🤞) for the axis environment.



In [70]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent = bytes_sent.set_index("benchmark").loc[throughputs["files"]].reset_index()
bytes_sent= bytes_sent.query("cluster_batch_size == 128")
bytes_sent["files"] = bytes_sent["benchmark"]
bytes_sent["benchmark"] = bytes_sent["benchmark"].apply(lambda s: "not-alternating-non-optimistic" if "not-alternating-non-optimistic" in s else (
    "alternating-non-optimistic" if "alternating-non-optimistic" in s else "alternating-optimistic"
))
bytes_sent["cluster_num_peers"] = bytes_sent["num_peers"] /  bytes_sent["number_of_clusters"] 
bytes_sent.query("benchmark == 'not-alternating-non-optimistic' and alternating == False").to_json(f"outputs/{notebook_name}/bytes_sent.json",orient="records")

for (group,df) in bytes_sent.groupby(['cluster_num_peers']): 
    # display(df)
    # continue
    role_df = get_role_df(df, role=2)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"benchmark"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    cluster_num_peers = group[0]
    print(f"Cluster Num Peers: {cluster_num_peers}")
    
     


    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role2-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
        
    )
     

    role_df = get_role_df(df, role=1)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"benchmark"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role1-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
    )


/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Cluster Num Peers: 3.0
[np.str_('(512,2,alternating-non-optimistic)'), np.str_('(512,2,alternating-optimistic)'), np.str_('(512,2,not-alternating-non-optimistic)'), np.str_('(512,4,alternating-non-optimistic)'), np.str_('(512,4,alternating-optimistic)'), np.str_('(512,4,not-alternating-non-optimistic)'), np.str_('(512,6,alternating-non-optimistic)'), np.str_('(512,6,alternating-optimistic)'), np.str_('(512,6,not-alternating-non-optimistic)'), np.str_('(4096,2,alternating-non-optimistic)'), np.str_('(4096,2,alternating-optimistic)'), np.str_('(4096,2,not-alternating-non-optimistic)'), np.str_('(4096,4,alternating-non-optimistic)'), np.str_('(4096,4,alternating-optimistic)'), np.str_('(4096,4,not-alternating-non-optimistic)'), np.str_('(4096,6,alternating-non-optimistic)'), np.str_('(4096,6,alternating-optimistic)'), np.str_('(4096,6,not-alternating-non-optimistic)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 e

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



entering extended mode

(./outputs/03-geobft-alternating/geobft-spread-over-azs-bytes-sent-role1-3.0.te
x
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/l

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



entering extended mode

(./outputs/03-geobft-alternating/geobft-spread-over-azs-bytes-sent-role2-6.0.te
x
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/l

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:372: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



[np.str_('(512,2,alternating-non-optimistic)'), np.str_('(512,2,alternating-optimistic)'), np.str_('(512,2,not-alternating-non-optimistic)'), np.str_('(512,4,alternating-non-optimistic)'), np.str_('(512,4,alternating-optimistic)'), np.str_('(512,4,not-alternating-non-optimistic)'), np.str_('(512,6,alternating-non-optimistic)'), np.str_('(512,6,alternating-optimistic)'), np.str_('(512,6,not-alternating-non-optimistic)'), np.str_('(4096,2,alternating-non-optimistic)'), np.str_('(4096,2,alternating-optimistic)'), np.str_('(4096,2,not-alternating-non-optimistic)'), np.str_('(4096,4,alternating-non-optimistic)'), np.str_('(4096,4,alternating-optimistic)'), np.str_('(4096,4,not-alternating-non-optimistic)'), np.str_('(4096,6,alternating-non-optimistic)'), np.str_('(4096,6,alternating-optimistic)'), np.str_('(4096,6,not-alternating-non-optimistic)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extend

In [71]:
res = {}
array_length = 1000
indices = throughputs.files.copy()
throughputs = process_throughput_benchmarks(benchmarks)


for benchmark in benchmarks: 

    received = np.zeros(array_length)
    response = np.zeros(array_length)
    latency = np.zeros(array_length)
    num_leaders = 0 
    for log in glob(f"{benchmark}/*.log"): 
        
        with open(log, 'r') as f:
            lines = [line for line in f.read().splitlines() if "client request" in line]

        if len(lines) != 2*array_length : 
            continue
        if lines : 
           
            for line in lines : 
                nanoseconds = line.split(" ")[-1]
                if "Received" in line: 
                    rc = int(line.split(" ")[3])
                    # print(line)
                    received[rc] = int(nanoseconds)
                    rc += 1
                elif "Response" in line: 
                    rp = int(line.split(" ")[3])
                    response[rp] = int(nanoseconds)
                    rp += 1
                else: 
                    raise ValueError("Unexpected line")
            
            latency += response - received
            num_leaders+=1
    latency = latency / num_leaders
    latency_ms = latency * 1e-6
    max_latency = np.max(latency_ms)
    avg_latency = np.median(latency_ms)
    min_latency = np.min(latency_ms)
    throughputs.loc[throughputs['benchmark'] == benchmark, 'latency_ms'] = avg_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'max_latency_ms'] = max_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'min_latency_ms'] = min_latency
    
    if min_latency < 0 : 
        raise ValueError("Negative latency detected")
    print(f"Benchmark: {benchmark}, Max Latency: {max_latency}, Avg Latency: {avg_latency}, Min Latency: {min_latency}")
        

Benchmark: ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251029151350-cb64-v512/, Max Latency: 174.058112, Avg Latency: 107.54009599999999, Min Latency: 41.239893333333335
Benchmark: ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251029152215-cb64-v4096/, Max Latency: 221.87758933333333, Avg Latency: 170.93629866666663, Min Latency: 92.20125866666666
Benchmark: ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251029152351-cb128-v512/, Max Latency: 142.95957333333334, Avg Latency: 86.10257066666665, Min Latency: 31.602858666666666
Benchmark: ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251029152531-cb128-v4096/, Max Latency: 277.885184, Avg Latency: 194.71127466666667, Min Latency: 73.50459733333332
Benchmark: ../../aws/benchmark/out/geobft/final-geobft-alternating-opt/20251029152716-cb256-v512/, Max Latency: 184.70344533333335, Avg Latency: 89.31479466666666, Min Latency: 43.10267733333333
Benchmark: ../../aws/benchmark/out/g

In [72]:
throughputs.set_index("benchmark", inplace=True)
throughputs = throughputs.loc[indices].reset_index()
throughputs["files"] = throughputs["benchmark"]
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "not-alternating-non-optimistic" if "not-alternating-non-optimistic" in s else (
    "alternating-non-optimistic" if "alternating-non-optimistic" in s else "alternating-optimistic"
))

In [73]:
throughputs = add_combined_column(throughputs, ['number_of_clusters', 'vallen',"benchmark"], 'num_clusters_vallen')
throughputs["cluster_num_peers"] = throughputs["num_peers"] /  throughputs["number_of_clusters"]
throughputs = throughputs.query("cluster_batch_size < 256")
def generate_latency_plot(throughputs, num_peers,vallen):
    xs = []
    ys = []
    cats = []

    for (group, df) in throughputs.query("cluster_num_peers == @num_peers and vallen == @vallen").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["cluster_batch_size", "benchmark", "cluster_num_peers"])
        cats.append(num_clusters_vallen)
        xs.append(sorteddf["cluster_batch_size"].tolist())
        ys.append(sorteddf["latency_ms"].tolist())

    tikz_plot_latency = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Cluster Batch Size",
        ylabel="Latency (ms)",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency-{num_peers}peers-{vallen}bytes.tex",
    )
    tikz_plot_latency.save()
    tikz_plot_latency.compile(output_dir=f"outputs/{notebook_name}/")

    return tikz_plot_latency

generate_latency_plot(throughputs, 6,512)
generate_latency_plot(throughputs, 6,4096)
generate_latency_plot(throughputs, 3,512)
generate_latency_plot(throughputs, 3,4096)

#     throughputs,
#     y_col="latency_ms",
#     x_col="num_peers",
#     cat_col="vallen",
#     filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency.tex",
#     xlabel="Number of Peers",
#     ylabel="Latency (ms)",
#     output_dir=f"outputs/{notebook_name}/"
# )

[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/03-geobft-alternating/geobft-spread-over-azs-latency-6peers-512bytes
.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/sta

In [74]:
#rename every file in output_dir=f"outputs/{notebook_name}/"
import os
output_dir=f"outputs/{notebook_name}/"
for filename in os.listdir(output_dir):
    if filename.endswith(".tex") or filename.endswith(".pdf"):
        new_filename = filename.replace("geobft-spread-over-azs","alternating-geobft-spread-over-azs")
        os.rename(os.path.join(output_dir, filename), os.path.join(output_dir, new_filename))
        
# remove log and aux 
for filename in os.listdir(output_dir):
    if filename.endswith(".log") or filename.endswith(".aux"):
        os.remove(os.path.join(output_dir, filename))

In [75]:
import datetime
import zipfile

timestamp=datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

with zipfile.ZipFile(f'outputs/{notebook_name}/notebook_files_{timestamp}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(f'outputs/{notebook_name}'):
        for file in files:
            if not file.endswith('.zip'):
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                                           os.path.join(f'outputs/{notebook_name}', '..')))

In [76]:

s = ""
for filename in os.listdir(output_dir):
    if filename.endswith(".tex"):
        tex_content = "%"+filename+"\n"
        with open(os.path.join(output_dir, filename), 'r') as f:
            for line in f:
                tex_content += line
                
            

        s+= "\n"+tex_content+"\n"

with open(f"outputs/{notebook_name}/figures.tex", 'w') as f:
    f.write(s)